# GOV-01 V2 — Final protected-test evaluation

**Purpose:** make the one final, honest measurement of the selected V2-E8 EfficientNetB0 multiclass model on the protected test split.

This notebook is evaluation only: it does **not** train, tune, alter data, alter labels, or save a new model. The protected test must never be used to choose a new setting. A small completion marker is saved in Google Drive after a successful evaluation to prevent accidental repeat use.

Run each code cell once, in order. Download and keep the final ZIP outside the Git repository.

In [ ]:
# Cell 1 — imports and Google Drive connection
from pathlib import Path
import json
import shutil
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from google.colab import drive, files
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support

drive.mount('/content/drive')
print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

In [ ]:
# Cell 2 — final-evaluation paths and safety gate
DRIVE_ROOT = Path('/content/drive/MyDrive/GOV-01')
ARCHIVE_PATH = DRIVE_ROOT / 'v2' / 'multiclass_final.zip'
MODEL_PATH = DRIVE_ROOT / 'v2_e8_checkpoint' / 'v2_e8_efficientnetb0_low_lr_best.keras'
TEST_AUDIT_MARKER = DRIVE_ROOT / 'v2_final_protected_test_evaluation_completed.json'

DATA_DIR = Path('/content/data/processed/v2/multiclass_final')
TEST_DIR = DATA_DIR / 'protected_test'
OUTPUT_DIR = Path('/content/v2_final_test_output')

CLASS_NAMES = ['crack', 'manhole_cover', 'normal_asphalt', 'pothole', 'repaired_road', 'speed_bump', 'unpaved_road']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EXPECTED_TEST_IMAGE_COUNT = 1654

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Missing V2 dataset ZIP in Google Drive: {ARCHIVE_PATH}')
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'Missing selected V2-E8 model in Google Drive: {MODEL_PATH}')
if TEST_AUDIT_MARKER.exists():
    raise RuntimeError(
        f'Protected test was already evaluated. Do not run it again.\n'
        f'Recorded result: {TEST_AUDIT_MARKER}'
    )

print('Selected model:', MODEL_PATH)
print('Final test image count expected:', EXPECTED_TEST_IMAGE_COUNT)
print('This is the one-time protected-test evaluation. No training will occur.')

In [ ]:
# Cell 3 — extract the persistent dataset into this temporary Colab runtime only when needed.
if not TEST_DIR.is_dir():
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR.parent)
    print('Extracted dataset to:', DATA_DIR)
else:
    print('Dataset already available in this runtime:', DATA_DIR)

actual_count = sum(1 for path in TEST_DIR.rglob('*') if path.is_file())
if actual_count != EXPECTED_TEST_IMAGE_COUNT:
    raise RuntimeError(f'Protected test has {actual_count} files; expected {EXPECTED_TEST_IMAGE_COUNT}. Stop and inspect the archive.')
print(f'Protected test verified: {actual_count} images')

In [ ]:
# Cell 4 — load the protected test exactly once, without shuffling.
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, label_mode='int', class_names=CLASS_NAMES, image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False
)
if test_ds.class_names != CLASS_NAMES:
    raise RuntimeError('Protected-test class order differs from the approved V2 label order.')
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)
print('Protected-test class order:', CLASS_NAMES)
print('No train or validation dataset has been loaded in this notebook.')

In [ ]:
# Cell 5 — one-time final test evaluation. This cell does not train or modify the model.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model = tf.keras.models.load_model(MODEL_PATH)
y_test = np.concatenate([labels.numpy() for _, labels in test_ds])
probabilities = model.predict(test_ds, verbose=1)
predictions = probabilities.argmax(axis=1)

precision, recall, f1_values, support = precision_recall_fscore_support(
    y_test, predictions, labels=np.arange(len(CLASS_NAMES)), zero_division=0
)
final_metrics = {
    'experiment': 'V2 final protected-test evaluation',
    'selected_model': 'V2-E8 EfficientNetB0 low-learning-rate continuation',
    'split': 'protected_test',
    'training_or_tuning_performed': False,
    'image_count': int(len(y_test)),
    'accuracy': float(accuracy_score(y_test, predictions)),
    'macro_f1': float(f1_score(y_test, predictions, average='macro')),
    'per_class': {
        class_name: {
            'precision': float(precision[index]), 'recall': float(recall[index]),
            'f1': float(f1_values[index]), 'support': int(support[index]),
        } for index, class_name in enumerate(CLASS_NAMES)
    },
}
(OUTPUT_DIR / 'v2_final_protected_test_metrics.json').write_text(json.dumps(final_metrics, indent=2), encoding='utf-8')
report = classification_report(y_test, predictions, target_names=CLASS_NAMES, digits=4, zero_division=0)
(OUTPUT_DIR / 'v2_final_protected_test_classification_report.txt').write_text(report, encoding='utf-8')
print('FINAL protected-test accuracy:', f"{final_metrics['accuracy']:.4f}")
print('FINAL protected-test macro F1:', f"{final_metrics['macro_f1']:.4f}")
print(report)

matrix = confusion_matrix(y_test, predictions, labels=np.arange(len(CLASS_NAMES)))
fig, axis = plt.subplots(figsize=(10, 8)); image = axis.imshow(matrix, cmap='Blues')
axis.set_xticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha='right')
axis.set_yticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES)
axis.set_xlabel('Predicted label'); axis.set_ylabel('Actual label'); axis.set_title('V2 final protected-test confusion matrix')
threshold = matrix.max() / 2 if matrix.max() else 0
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axis.text(column, row, str(matrix[row, column]), ha='center', va='center', color='white' if matrix[row, column] > threshold else 'black')
fig.colorbar(image, ax=axis); fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'v2_final_protected_test_confusion_matrix.png', dpi=150)
plt.show()

# The marker records that this one-time final test has been used. It contains metrics only, not data or a model.
TEST_AUDIT_MARKER.write_text(json.dumps(final_metrics, indent=2), encoding='utf-8')
print('Final-test completion marker saved in Drive. Do not rerun this notebook.')

In [ ]:
# Cell 6 — package the final evidence for browser download. Do not commit this ZIP or its contents.
required_files = [
    OUTPUT_DIR / 'v2_final_protected_test_metrics.json',
    OUTPUT_DIR / 'v2_final_protected_test_classification_report.txt',
    OUTPUT_DIR / 'v2_final_protected_test_confusion_matrix.png',
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing final-test evidence: {missing}')
archive_path = shutil.make_archive('/content/v2_final_protected_test_output', 'zip', OUTPUT_DIR)
print('Created browser-download package:', archive_path)
print('Keep this ZIP outside the Git repository. Do not commit it.')
files.download(archive_path)